In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

# Project directory
PROJECT_DIR = Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"

# Output files stay in your project directory
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "output" 

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_FILES = {
    2019: RAW_DIR / "AnnualisedEntryExit_2019.xlsx",
    2022: RAW_DIR / "AC2022_AnnualisedEntryExit.xlsx",
    2025: RAW_DIR / "AC2025_AnnualisedEntryExit_public.xlsx",
}

In [2]:
def load_data(file_path, sheet_name=0, skiprows=0,
              delete_first_row=False, drop_last_n=0, drop_ids=None):

    file_path = Path(file_path)

    if not file_path.exists():
        print(f"File not found: {file_path}")
        return None

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            skiprows=skiprows
        )

        # Delete the first data row if required
        if delete_first_row:
            df = df.drop(index=0).reset_index(drop=True)

    elif file_path.suffix.lower() == ".csv":
        df = pd.read_csv(file_path)

    else:
        raise ValueError(f"Unsupported file format: {file_path.suffix}")

    # Clean column names
    df.columns = (
        df.columns.astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    
    # Delete rows based on ID in the second column
    if drop_ids is not None:
        df = df[~df.iloc[:, 1].isin(drop_ids)]

    df = df.reset_index(drop=True)

    return df

In [3]:
YEAR_SHEETS = {
    2019: "Annualised",
    2022: 0,
    2025: 0,
}

YEAR_SKIPROWS = {
    2019: 6,
    2022: 6,
    2025: 5,
}

YEAR_DELETE_FIRST_ROW = {
    2019: False,
    2022: False,
    2025: True,
}

YEAR_DROP_IDS = {
    2019: [618],
    2022: [618],
    2025: [618],
}

data_by_year = {}

for year, file_path in YEAR_FILES.items():

    df = load_data(
        file_path,
        sheet_name=YEAR_SHEETS[year],
        skiprows=YEAR_SKIPROWS[year],
        delete_first_row=YEAR_DELETE_FIRST_ROW[year],
        drop_ids=YEAR_DROP_IDS[year]
    )

    if df is not None:
        data_by_year[year] = df
        print(f"{year}: {df.shape[0]} rows, {df.shape[1]} columns")

2019: 448 rows, 15 columns
2022: 468 rows, 19 columns
2025: 468 rows, 19 columns


In [4]:
# for year, df in data_by_year.items():
#     print(f"\n{year}")
#     print([repr(col) for col in df.columns])

In [5]:
lu_data = {}

for year, df in data_by_year.items():

    if "Mode" not in df.columns:
        print(f"Warning: Mode column missing in {year}")
        continue

    lu = df[
        df["Mode"].astype(str).str.strip().str.upper() == "LU"
    ].copy()

    lu_data[year] = lu

    # print(f"{year}: {len(lu)} London Underground records")
    
    # print(f"\n========== {year} ==========")

    # # Basic information
    # print(f"Rows: {len(lu)}")
    # print(f"Columns: {len(lu.columns)}")

    # # Check duplicate IDs in the second column
    # id_col = lu.columns[1]
    # print(f"ID column: {id_col}")
    # print(f"Duplicate IDs: {df[id_col].duplicated().sum()}")

    # # Display first and last 5 rows
    # print("\nFirst 5 rows:")
    # display(lu.head())

    # print("\nLast 5 rows:")
    # display(lu.tail())

In [6]:

def standardise_annual_data(df, year):

    df = df.copy()

    df.columns = (
        df.columns.astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Year-specific column mapping
    if year == 2019:

        rename_dict = {
            "NLC": "MNLC",
            "ASC": "MASC",
            "entries": "MTT_Entry",
            "entries.1": "Friday_Entry",
            "entries.2": "Saturday_Entry",
            "entries.3": "Sunday_Entry",
            "exits": "MTT_Exit",
            "exits.1": "Friday_Exit",
            "exits.2": "Saturday_Exit",
            "exits.3": "Sunday_Exit",
            "en/ex": "Annualised_EnEx"
        }

        df = df.rename(columns=rename_dict)

        df["Monday_Entry"] = np.nan
        df["TWT_Entry"] = np.nan
        df["Monday_Exit"] = np.nan
        df["TWT_Exit"] = np.nan

        df["Weekly_EnEx"] = np.nan
        df["12week_EnEx"] = np.nan

    elif year == 2022:

        rename_dict = {
            "Entries": "Monday_Entry",
            "Entries.1": "TWT_Entry",
            "Entries.2": "Friday_Entry",
            "Entries.3": "Saturday_Entry",
            "Entries.4": "Sunday_Entry",
            "Exits": "Monday_Exit",
            "Exits.1": "TWT_Exit",
            "Exits.2": "Friday_Exit",
            "Exits.3": "Saturday_Exit",
            "Exits.4": "Sunday_Exit",
            "En/Ex": "Weekly_EnEx",
            "En/Ex.1": "12week_EnEx",
            "En/Ex.2": "Annualised_EnEx"
        }

        df = df.rename(columns=rename_dict)

    elif year == 2025:

        rename_dict = {
            "Monday": "Monday_Entry",
            "Midweek (Tue-Thu)": "TWT_Entry",
            "Friday": "Friday_Entry",
            "Saturday": "Saturday_Entry",
            "Sunday": "Sunday_Entry",
            "Monday.1": "Monday_Exit",
            "Midweek (Tue-Thu).1": "TWT_Exit",
            "Friday.1": "Friday_Exit",
            "Saturday.1": "Saturday_Exit",
            "Sunday.1": "Sunday_Exit",
            "Weekly": "Weekly_EnEx",
            "12-week": "12week_EnEx",
            "Annualised": "Annualised_EnEx"
        }

        df = df.rename(columns=rename_dict)

    else:
        raise ValueError(f"Unsupported year: {year}")

    # Convert passenger counts to numeric
    numeric_columns = [
        "Monday_Entry", "TWT_Entry", "MTT_Entry",
        "Friday_Entry", "Saturday_Entry", "Sunday_Entry",
        "Monday_Exit", "TWT_Exit", "MTT_Exit",
        "Friday_Exit", "Saturday_Exit", "Sunday_Exit",
        "Weekly_EnEx", "12week_EnEx", "Annualised_EnEx"
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str)
                .str.replace(",", "", regex=False)
                .str.strip(),
                errors="coerce"
            )

    # Calculate MTT average for 2022 and 2025
    if year >= 2022:
        df["MTT_Entry"] = (
            df["Monday_Entry"] + 3 * df["TWT_Entry"]
        ) / 4

        df["MTT_Exit"] = (
            df["Monday_Exit"] + 3 * df["TWT_Exit"]
        ) / 4

    df["Year"] = year

    final_columns = [
        "Year", "Mode", "MNLC", "MASC", "Station",
        "Coverage", "Source",
        "MTT_Entry", "Friday_Entry", "Saturday_Entry", "Sunday_Entry",
        "MTT_Exit", "Friday_Exit", "Saturday_Exit", "Sunday_Exit",
        "Annualised_EnEx"
    ]

    # Ensure all years have the same columns
    for col in final_columns:
        if col not in df.columns:
            df[col] = np.nan

    return df[final_columns]

In [7]:
standardised_data = {}

for year, df in lu_data.items():

    standardised_data[year] = standardise_annual_data(df, year)

    print(
        f"{year}: {standardised_data[year].shape[0]} rows, "
        f"{standardised_data[year].shape[1]} columns"
    )

2019: 269 rows, 16 columns
2022: 269 rows, 16 columns
2025: 269 rows, 16 columns


In [8]:

# ==========================================
# 2. Combine all years into one dataset
# ==========================================

combined_df = pd.concat(
    standardised_data.values(),
    ignore_index=True
)

# Sort by year and station
combined_df = combined_df.sort_values(
    ["Year", "Station"]
).reset_index(drop=True)


# ==========================================
# 3. Create data audit summary
# ==========================================

audit_records = []

for year in data_by_year:

    raw_rows = len(data_by_year[year])

    lu_rows = len(lu_data[year]) if year in lu_data else 0

    processed_rows = (
        len(standardised_data[year])
        if year in standardised_data
        else 0
    )

    audit_records.append({
        "Year": year,
        "Raw_Rows": raw_rows,
        "LU_Rows": lu_rows,
        "Processed_Rows": processed_rows,
        "Raw_Columns": data_by_year[year].shape[1],
        "Final_Columns": (
            standardised_data[year].shape[1]
            if year in standardised_data
            else 0
        )
    })

audit_df = pd.DataFrame(audit_records)


# ==========================================
# 4. Save final datasets
# ==========================================

# Save consolidated dataset
combined_df.to_csv(
    PROCESSED_DIR / "london_underground_clean.csv",
    index=False
)

# Save individual yearly datasets
for year, df in standardised_data.items():

    df.to_csv(
        PROCESSED_DIR / f"{year}_london_underground_clean.csv",
        index=False
    )



# ==========================================
# 5. Final validation
# ==========================================

print("\nData preprocessing completed.")

print(f"Combined dataset: {combined_df.shape}")

print(f"Total records: {len(combined_df)}")

print("\nRecords by year:")
print(combined_df.groupby("Year").size())

print("\nAudit summary:")
print(audit_df)

print("\nFinal columns:")
print(combined_df.columns.tolist())



Data preprocessing completed.
Combined dataset: (807, 16)
Total records: 807

Records by year:
Year
2019    269
2022    269
2025    269
dtype: int64

Audit summary:
   Year  Raw_Rows  LU_Rows  Processed_Rows  Raw_Columns  Final_Columns
0  2019       448      269             269           15             16
1  2022       468      269             269           19             16
2  2025       468      269             269           19             16

Final columns:
['Year', 'Mode', 'MNLC', 'MASC', 'Station', 'Coverage', 'Source', 'MTT_Entry', 'Friday_Entry', 'Saturday_Entry', 'Sunday_Entry', 'MTT_Exit', 'Friday_Exit', 'Saturday_Exit', 'Sunday_Exit', 'Annualised_EnEx']
